# 립리딩 배포 모델 — 굽기

**이 노트북은 실험용이 아니다.** 실험은 `colab_pipeline.ipynb`에서 하고,
여기서는 확정된 설정으로 배포 모델만 만든다. 위에서부터 순서대로 실행하면 된다.

| | 실험 노트북 | 이 노트북 |
|---|---|---|
| 목적 | 조건을 바꿔가며 측정 | 확정 설정으로 배포본 생산 |
| 검증 화자 | 한 명씩 빼고 잰다 | **빼지 않는다.** 전원으로 학습 |
| 저장 기준 | 검증 최고점 | 검증이 없으므로 에폭 고정(50) |
| 산출물 | 측정값 | `.pt` 3개 + 문구 목록 |
| 몽키패치 | 있음 | **없음** |

## 배포 설정 (2026-08-25 확정)

```
크롭      192x96 · 닮음 변환 정렬
프레임    60
모델      3D-Conv + ResNet-18 + BiGRU(hidden 300, 2층) + 평균풀링 헤드
학습      80에폭 교차검증으로 측정 · 배포는 50에폭 고정
시드      42 · 1 · 7 세 개를 굽고 추론에서 확률을 평균한다
```

## 성능

**교차검증 평균 0.621** (8화자 3시드 · 마지막 에폭 기준). "처음 보는 화자"에
대한 값이다. 학습에 포함된 화자면 더 높다.

**최종 모델 자체의 성능은 측정된 적이 없다.** 시험 집합을 따로 두지 않았기
때문이다. 서류에는 교차검증 값을 쓰고 그 뜻을 밝힐 것.


## 1. 셋업

드라이브 마운트 · 저장소 받기 · npy 규격 검증 · 로컬 복사 · 매니페스트 생성을
한 번에 한다. 끊겨도 다시 돌리면 이어받는다.

**규격이 다르면 여기서 멈춘다.** 개수만 맞고 규격이 옛것이면 로컬 사본을 지우고
다시 받는다.

### 실험 세션과 동시에 돌려도 된다

드라이브는 한 계정이 여러 세션에서 마운트할 수 있다. 부딪히는 건 **같은 파일에
동시에 쓸 때**뿐이라, 배포는 쓰는 대상을 전부 나눠 놨다.

| | 실험 노트북 | 이 노트북 |
|---|---|---|
| 매니페스트 | `manifest_f60.csv` | `manifest_release.csv` |
| 로컬 사본 | `/content/data_f60` | `/content/data_release` |
| 체크포인트 | `cv192_*` `no5_192_*` … | `release192_*` |
| 결과 JSON | `results/*.json` | 없음 |
| wandb | 씀 | 안 씀 |

`processed_align/`은 양쪽 다 **읽기만** 한다.

**다만 4.1GB 복사를 두 세션이 동시에 하면 드라이브가 느려지고 Errno 107이 잦아진다.**
한쪽 복사가 끝난 뒤 다른 쪽을 시작하는 편이 결국 빠르다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
# 크롭 규격마다 폴더가 다르다. 섞으면 백본이 첫 배치에서 죽는다.
#   processed_f60     112x80 · 축 정렬 상자        (옛 규격)
#   processed_align   192x96 · 닮음 변환 정렬      (2026-08-24~, 현재)
PROCESSED = DRIVE_ROOT / "processed_align"

n_drive = len(list(PROCESSED.glob("*.npy")))
print("Drive", PROCESSED.name + ":", n_drive, "개")
assert n_drive == 1238, f"{PROCESSED.name}이 1238개가 아니다"

os.chdir("/content")
for junk in Path("/content").glob("*REPO_DIR*"):
    shutil.rmtree(junk, ignore_errors=True)
    print("잘못 만들어진 폴더 삭제:", junk.name)

REPO = "/content/hanium-lipreading"
REPO_DIR = Path(REPO)
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "develop"], check=True)
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "-b", "develop",
                    "https://github.com/HumanRhoid/hanium-lipreading.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
# predict.py가 mediapipe를 쓴다. 코랩 기본 이미지에는 없다.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "mediapipe"], check=True)

# 실험 노트북은 manifest_f60.csv에 쓴다. 같은 파일을 두 세션이 동시에 쓰면
# 한쪽이 읽는 중에 다른 쪽이 덮어써 학습이 깨진다. 배포는 제 이름을 쓴다.
from scripts.build_manifest import build
MANIFEST = DRIVE_ROOT / "manifest_release.csv"
build(processed_dir=PROCESSED, manifest_path=MANIFEST)

import numpy as np
from src.ml.models.backbone import LipReadingBackbone
from src.ml.preprocess.normalize import FIXED_FRAME_COUNT

# (프레임, 높이, 너비, 채널). 192x96 이면 (60, 96, 192, 3)이다.
WANT = (FIXED_FRAME_COUNT, LipReadingBackbone.input_height,
        LipReadingBackbone.input_width, 3)


def npy_shape(folder):
    """폴더의 npy 한 장을 열어 규격을 본다. 비어 있으면 None."""
    files = sorted(folder.glob("*.npy")) if folder.exists() else []
    return np.load(files[0]).shape if files else None

drive_shape = npy_shape(PROCESSED)
assert drive_shape == WANT, f"드라이브 규격 불일치: {PROCESSED.name} {drive_shape} · 기대 {WANT}"

# 코랩 세션마다 /content가 따로라 경로가 같아도 안 부딪힌다. 다만 한 세션에서
# 두 노트북을 돌리는 실수를 막으려고 이름을 나눠 둔다.
TRAIN_ROOT = Path("/content/data_release")
LOCAL = TRAIN_ROOT / "processed"
started = time.time()

# 개수만 보면 옛 규격이 같은 개수로 남아 있을 때 복사를 건너뛴다. 실제로 그
# 사고가 났다(로컬에 112x80 1238개가 남아 재복사가 안 됨). 규격까지 본다.
local_shape = npy_shape(LOCAL)
if local_shape is not None and local_shape != WANT:
    print("옛 규격 로컬 사본 제거:", local_shape)
    shutil.rmtree(TRAIN_ROOT, ignore_errors=True)

# Drive FUSE는 파일 1238개를 한 번에 긁으면 자주 끊긴다(Errno 107 Transport
# endpoint is not connected). copytree는 한 장만 실패해도 통째로 무너지므로
# 직접 채우고 실패분을 재시도한다. 끊겨도 다시 돌리면 이어받는다.
#
# 있는지만 보면 안 된다. 끊길 때 만들어지다 만 0바이트 파일이 남는데 그것도
# exists()는 참이라 건너뛴다. 모든 npy가 같은 규격이라 크기도 같으므로
# 원본 한 장의 크기와 대조해 덜 받은 것을 가려낸다.
LOCAL.mkdir(parents=True, exist_ok=True)
sources = sorted(PROCESSED.glob("*.npy"))
REF_SIZE = sources[0].stat().st_size

def needs_copy(src):
    dst = LOCAL / src.name
    return not dst.exists() or dst.stat().st_size != REF_SIZE

def fetch(src):
    """성공하면 None, 실패하면 그 원본을 돌려준다."""
    try:
        shutil.copy2(src, LOCAL / src.name)
        return None
    except OSError:
        return src

# 드라이브는 한 장씩 받으면 지연이 대부분이라 동시에 받는 편이 몇 배 빠르다.
todo = [f for f in sources if needs_copy(f)]
for attempt in range(1, 4):
    if not todo:
        break
    print(f"복사 {len(todo)}개 · {len(todo) * REF_SIZE / 1e9:.1f} GB (시도 {attempt})")
    t0, done, failed = time.time(), 0, []
    with ThreadPoolExecutor(16) as pool:
        for result in pool.map(fetch, todo):
            done += 1
            if result is not None:
                failed.append(result)
            if done % 200 == 0:
                sec = max(time.time() - t0, 0.001)
                print(f"  {done}/{len(todo)} · {sec:.0f}초 · "
                      f"{done * REF_SIZE / 1e6 / sec:.0f} MB/s")
    todo = failed
    if todo:
        print("  실패", len(todo), "개 - 재시도")
if todo:
    raise RuntimeError(f"복사 실패 {len(todo)}개. 드라이브 마운트가 끊긴 것이니 "
                       "런타임을 다시 시작하고 이 셀을 다시 돌릴 것. 받은 것은 남는다.")

n_local = len(list(LOCAL.glob("*.npy")))
bad = [p.name for p in LOCAL.glob("*.npy") if p.stat().st_size != REF_SIZE]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, f"로컬 복사 불완전 · 크기 이상 {len(bad)}개"
assert npy_shape(LOCAL) == WANT, "복사 후에도 규격이 안 맞는다"
print("규격 확인", WANT)

## 2. 오염 검사

실험 노트북은 `LipReadingModel` 클래스를 런타임에 덮어쓰는 셀이 있다(랜드마크
융합). 그 커널에서 배포 모델을 구우면 구조가 다른 `.pt`가 나오고 `predict.py`가
못 읽는다. 2026-08-26에 실제로 그 사고가 났다.

**이 노트북을 새 런타임에서 열었다면 통과한다.** 실패하면 런타임을 다시 시작할 것.

In [ ]:
import torch
from src.ml.models import LipReadingModel
from src.ml.preprocess.normalize import FIXED_FRAME_COUNT, TARGET_HEIGHT, TARGET_WIDTH

def assert_clean():
    """실험 셀이 갈아끼운 것이 남아 있으면 멈춘다.

    파라미터 수로 재면 안 된다. hidden_dim 기본값(256)과 실험값(300)이 달라
    멀쩡한 모델도 13.59M로 나온다. 갈아끼웠는지를 직접 본다.
    """
    from src.ml.models.lip_reading_model import LipReadingModel
    from src.ml.preprocess.augmentation import VideoAugmentation
    from src.ml.training.dataset import LipReadingDataset
    from src.ml.training import train as T

    original = {
        "LipReadingModel.__init__": LipReadingModel.__init__,
        "LipReadingModel.forward": LipReadingModel.forward,
        "LipReadingDataset.__getitem__": LipReadingDataset.__getitem__,
        "VideoAugmentation.__call__": VideoAugmentation.__call__,
        "run_epoch": T.run_epoch,
        "collect_errors": T.collect_errors,
    }
    dirty = {k: v.__qualname__ for k, v in original.items()
             if v.__qualname__ != k.split(".")[-1] and v.__qualname__ != k}
    if dirty:
        raise AssertionError(
            "실험 패치가 남아 있다: " + ", ".join(f"{k} -> {v}" for k, v in dirty.items())
            + " · 런타임을 새로 켤 것"
        )
    print("패치 없음 · 깨끗함")


assert_clean()
_n = sum(p.numel() for p in LipReadingModel(num_classes=15, hidden_dim=300).parameters())
print(f"모델 {_n/1e6:.2f}M · 규격 {TARGET_WIDTH}x{TARGET_HEIGHT} · {FIXED_FRAME_COUNT}프레임")
print("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## 3. 화자 선택

기본은 **전원**이다. 특정 화자를 빼고 굽기로 결정했다면 `EXCLUDE`에 적는다.
빼는 판단은 실험 노트북의 `14. 화자 제외 실험` 결과로 한다. 근거 없이 빼지 말 것.

In [ ]:
import csv

EXCLUDE = []          # 예: ["s05"]. 비우면 전원.
NAME    = "release192"  # 체크포인트 이름 앞부분

manifest_release = MANIFEST
if EXCLUDE:
    rows = list(csv.DictReader(open(MANIFEST, encoding="utf-8")))
    keep = [r for r in rows if r["speaker_id"] not in EXCLUDE]
    assert keep, "전부 걸러졌다"
    manifest_release = DRIVE_ROOT / f"manifest_{NAME}.csv"
    with open(manifest_release, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["clip_path", "label_id", "label_text",
                                          "speaker_id", "take"])
        w.writeheader()
        w.writerows(keep)
    NAME = NAME + "_no" + "".join(s.replace("s", "") for s in EXCLUDE)
    print("제외", EXCLUDE, "·", len(rows), "->", len(keep), "클립")
else:
    print("전원 학습")

rows = list(csv.DictReader(open(manifest_release, encoding="utf-8")))
print("이름", NAME, "· 클립", len(rows),
      "· 화자", sorted({r["speaker_id"] for r in rows}),
      "· 문구", len({r["label_text"] for r in rows}), "개")

## 4. 학습

시드당 약 17분, 셋이면 **약 52분**이다. 이미 있는 체크포인트는 건너뛰므로
끊겨도 다시 돌리면 이어받는다.

에폭 50은 검증을 보지 않고 고를 수 있는 값이다. cv192 24런에서 `best_epoch`
중앙값 54, 정점 40, 포화 36이라 그 사이에 있다.

In [ ]:
cmd = (
    f"python scripts/train_release.py --seeds 42 1 7 --name {NAME}"
    f" --manifest {manifest_release} --data-root {TRAIN_ROOT}"
    f" --checkpoint-dir {DRIVE_CHECKPOINTS}"
)
print(cmd)
!{cmd}

## 5. 산출물 확인

`.pt` 안에 문구 목록과 프레임 수가 함께 들어 있어야 추론이 매니페스트 없이 돈다.
없으면 교차검증 체크포인트를 잘못 집은 것이다.

In [ ]:
import torch

RELEASE = sorted(DRIVE_CHECKPOINTS.glob(NAME + "_seed*.pt"))
assert RELEASE, f"{NAME}_seed*.pt 가 없다"
for p in RELEASE:
    s = torch.load(p, map_location="cpu")
    print(f"{p.name}  {p.stat().st_size/1e6:.0f}MB  seed {s['seed']} · "
          f"{s['epochs']}에폭 · {s['clips']}클립 · {s['frames']}프레임 · "
          f"화자 {len(s['speakers'])}명")
    assert "labels" in s and s["frames"] == FIXED_FRAME_COUNT
print()
print("문구", len(s["labels"]), "개:", ", ".join(s["labels"]))

## 6. 추론 확인

**세 개를 모두 넘긴다.** 한 모델만 쓰면 시드 편차(0.0725) 때문에 같은 영상에도
답이 갈린다. `predict.py`는 확률을 평균한다.

전처리는 학습과 같은 함수(`vid2npy.process_video`)를 부른다. 여기서 한 줄이라도
달라지면 성능이 조용히 떨어진다.

In [ ]:
VIDEO = next(iter(sorted((DRIVE_ROOT / "raw").glob("*.mp4"))), None)
assert VIDEO is not None, "확인용 영상이 없다"
print("정답(파일명):", VIDEO.stem)

ckpts = " ".join(f'"{p}"' for p in RELEASE)
cmd = f'python scripts/predict.py "{VIDEO}" --checkpoint {ckpts} --verbose'
print(cmd)
!{cmd}

## 7. 전달

드라이브 `checkpoints/`의 `.pt` 세 개가 산출물이다. 함께 넘길 것:

- **사용법** — `python scripts/predict.py 영상.mp4 --checkpoint a.pt b.pt c.pt`
- **입력 조건** — 정면 얼굴, 입이 프레임 안에 들어올 것. mp4/avi/mov
- **성능** — 교차검증 평균 0.621 (처음 보는 화자 기준, 8화자 3시드)
- **한계** — 폐쇄형 15문구 전용. 목록 밖의 말은 15개 중 하나로 잘못 답한다
- **버전** — `git rev-parse --short HEAD` 값을 함께 적을 것

In [ ]:
!git -C /content/hanium-lipreading rev-parse --short HEAD
!git -C /content/hanium-lipreading log -1 --format="%cd %s" --date=short
print()
for p in RELEASE:
    print(p)